# 08. 카드 데이터가 5건 미만은 마스킹 된 것인지 확인

## 0. 준비

### 0-1. 라이브러리 불러오기와 그래프 기본 설정

In [31]:
from pathlib import Path          # 파일 경로 다루기 (윈도우/맥 상관없이 동작)
import numpy as np                # 숫자 계산
import pandas as pd               # 표(DataFrame) 다루기
import matplotlib.pyplot as plt   # 그래프 그리기
from IPython.display import display   # 표를 칸이 나뉜 표로 보여줌

# ── 한글 폰트 설정 (안 하면 그래프의 한글이 네모로 깨짐) ──
plt.rc('font', family='Malgun Gothic')     # 윈도우 기본 한글 폰트
plt.rc('axes', unicode_minus=False)        # 음수 부호(-)가 깨지는 것 방지

plt.rc('figure', figsize=(9, 4), dpi=110)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 40)

### 0-2. 경로 설정과 데이터 불러오기

In [32]:
# 01번 노트북과 같은 방식으로 프로젝트 맨 위 폴더를 찾는다.
ROOT = Path.cwd()
while not (ROOT / 'dataset').exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError('dataset 폴더를 찾을 수 없다. 노트북 위치를 확인한다.')
    ROOT = ROOT.parent
OUT = ROOT / 'notebooks' / 'preprocessed'

card1 = pd.read_parquet(OUT / 'card1.parquet')          # 데이터1: 날짜 x 지역 x 업종 x 성별 x 연령 x 시간대
card2 = pd.read_parquet(OUT / 'card2.parquet')          # 데이터2: 날짜 x 지역 x 업종 x 성별 x 연령 x 거주지
calendar = pd.read_parquet(OUT / 'calendar.parquet')    # 달력 (요일, 공휴일)
sales = pd.read_parquet(OUT / 'sales_seoul.parquet')    # 서울 상권분석 추정매출 (4장 외부 대조용)

for name, df in {'card1': card1, 'card2': card2, 'sales': sales}.items():
    print(f'{name:7s} {df.shape[0]:>10,}행 x {df.shape[1]:>2}열')

card1    1,044,710행 x 11열
card2    1,806,800행 x 11열
sales      343,167행 x 56열


### 0-3. 용어

| 말 | 뜻 |
|---|---|
| **칸** | `날짜 + 지역 + 업종 + 성별 + 연령 + 시간대` 조합 하나 (표의 한 행) |
| **계열** | 칸에서 날짜만 뺀 것. 184일치 시계열 하나 |
| **구멍** | 계열의 첫 관측일과 마지막 관측일 사이에 행이 없는 날 |

데이터2는 법인카드를 뺀 파일이라, 비교 조건을 맞추려고 데이터1도 개인 결제만 쓴다.

### 0-4. 개인 결제만 남기기

In [33]:
개인 = card1[card1['sex'] != '법인'].copy()      # 남성, 여성 결제
법인 = card1[card1['sex'] == '법인'].copy()
print(f'데이터1 전체 {len(card1):,}행 = 개인 {len(개인):,}행 + 법인 {len(법인):,}행')
print('데이터2 sex:', sorted(card2['sex'].unique()))

데이터1 전체 1,044,710행 = 개인 983,413행 + 법인 61,297행
데이터2 sex: ['남성', '여성']


## 1. 계열 x 날짜 행렬 만들기

### 1-1. 계열 번호와 날짜 번호 매기기

빈칸은 nan으로 만들기

In [34]:
S = ['region', 'industry', 'sex', 'age', 'time_gb']   # 계열을 정하는 5개 항목 (날짜 제외)

# 계열마다 0, 1, 2, ... 번호를 매긴다 (ngroup = group number)
개인['계열번호'] = 개인.groupby(S, observed=True).ngroup()

# 날짜마다 0, 1, 2, ... 번호를 매긴다 (7월 1일이 0번)
날짜목록 = np.sort(개인['date'].unique())
개인['날짜번호'] = 개인['date'].map({d: i for i, d in enumerate(날짜목록)})

계열수 = 개인['계열번호'].nunique()
날짜수 = len(날짜목록)
print(f'계열 {계열수:,}개 x {날짜수}일')

계열 9,837개 x 184일


### 1-2. 행렬 채우기
`계열수 x 184` 빈 표에 있는 값만 채운다. **남은 NaN = 그날 행이 없었다.**

In [35]:
# np.nan 으로 가득 찬 표를 만든다. nan = '값 없음'
M = np.full((계열수, 날짜수), np.nan)

# 데이터에 있는 칸만 값을 꽂아 넣는다
M[개인['계열번호'].values, 개인['날짜번호'].values] = 개인['cnt'].values

있음 = ~np.isnan(M)       # True = 그날 행이 있었다, False = 행이 없었다
print(f'표 크기 {M.shape}')
print(f'값이 있는 칸 {있음.sum():,}개 / 전체 {M.size:,}칸 = {있음.mean():.1%}')

# 예시로 한 계열을 꺼내 본다.
# 처음 30일 안에 값이 있으면서 빈 날도 섞여 있는 계열을 골라야 '구멍'이 눈에 보인다.
있는날수 = 있음[:, :30].sum(1)
후보 = np.where((있는날수 >= 12) & (있는날수 <= 24))[0]
예시 = int(후보[0])

이름표 = 개인.groupby(S, observed=True).size().reset_index()[S].iloc[예시]
print(f"\n예시 계열: {이름표['region']} {이름표['industry']} "
      f"{이름표['sex']} {이름표['age']} {이름표['time_gb']}  (처음 30일, nan = 행 없음)")
print(np.round(M[예시, :30], 0))

표 크기 (9837, 184)
값이 있는 칸 983,413개 / 전체 1,810,008칸 = 54.3%

예시 계열: 강남 LPG가스 남성 20대 00_05  (처음 30일, nan = 행 없음)
[ 5.  5. nan 18. nan nan  5.  5.  5. 12. nan nan nan 16. 11. 12.  6. nan
  6. nan  5. nan nan nan  6. nan  5.  6.  6. nan]


### 1-3. '구멍'의 정의

맨 앞과 맨 뒤가 빈 건 개업 전, 폐업 후라 제외한다. **첫 관측일과 마지막 관측일 사이**만 본다.

```
. . . 5 7 . 6 . . 9 8 . . .
└─앞─┘ └──── 이 구간만 ────┘ └뒤┘
```

In [36]:
# 계열마다 첫 관측일과 마지막 관측일의 위치를 찾는다
첫날 = 있음.argmax(1)                          # argmax는 True가 처음 나오는 위치를 준다
마지막날 = 날짜수 - 1 - 있음[:, ::-1].argmax(1)   # 뒤집어서 찾으면 마지막 위치

# 각 계열의 [첫날 ~ 마지막날] 구간만 True 로 표시한 표
열번호 = np.arange(날짜수)[None, :]
관측구간 = (열번호 >= 첫날[:, None]) & (열번호 <= 마지막날[:, None])

# 구멍 = 관측구간 안인데 값이 없는 칸
구멍 = 관측구간 & ~있음

print(f'관측구간 안의 날 총합 : {관측구간.sum():,}일')
print(f'그중 구멍            : {구멍.sum():,}일 ({구멍.sum() / 관측구간.sum():.1%})')

관측구간 안의 날 총합 : 1,566,501일
그중 구멍            : 583,088일 (37.2%)


## 2. 어느 날에는 존재하던 행이 다음 날 아예 사라지는 사례가 있는지 확인

### 2-1. 그런 사례는 몇 개인가?

In [37]:
구멍있는계열 = int((구멍.sum(1) > 0).sum())

# '오늘은 있는데 내일은 없음'이 몇 번 일어났는지
# 있음[:, :-1] = 오늘 있음, ~있음[:, 1:] = 내일 없음
오늘있고내일없음 = 있음[:, :-1] & ~있음[:, 1:] & 관측구간[:, 1:]

print(f'구멍이 있는 계열      : {구멍있는계열:,}개 / {계열수:,}개 ({구멍있는계열 / 계열수:.1%})')
print(f'구멍 난 날 총합       : {구멍.sum():,}일')
print(f'"오늘 있고 내일 없음" : {오늘있고내일없음.sum():,}회')

구멍이 있는 계열      : 7,073개 / 9,837개 (71.9%)
구멍 난 날 총합       : 583,088일
"오늘 있고 내일 없음" : 136,847회


### 2-2. 사라진 행들은 얼마나 지속되었는가?

In [38]:
# 구멍이 연속으로 이어진 덩어리를 찾아낸다.
# 방법: 0과 1로 된 줄의 앞뒤에 0을 붙이고 차분(diff)을 내면
#       1이 나오는 자리 = 덩어리 시작, -1이 나오는 자리 = 덩어리 끝
덩어리 = []
for i in range(계열수):
    g = 구멍[i]
    if not g.any():
        continue
    차분 = np.diff(np.concatenate(([0], g.astype(np.int8), [0])))
    시작들 = np.where(차분 == 1)[0]
    끝들 = np.where(차분 == -1)[0]
    for s, e in zip(시작들, 끝들):
        덩어리.append((i, s, e - s))

덩어리 = pd.DataFrame(덩어리, columns=['계열번호', '시작일번호', '길이'])
print(f'구멍 덩어리 {len(덩어리):,}개\n')

길이분포 = 덩어리['길이'].value_counts().sort_index()
표 = pd.DataFrame({
    '덩어리 수': [길이분포.get(k, 0) for k in [1, 2, 3, 4, 5]] + [int((덩어리['길이'] >= 6).sum())],
    '비율': [길이분포.get(k, 0) / len(덩어리) for k in [1, 2, 3, 4, 5]] + [(덩어리['길이'] >= 6).mean()],
}, index=['1일', '2일', '3일', '4일', '5일', '6일 이상'])
표['비율'] = 표['비율'].map('{:.1%}'.format)
display(표)

구멍 덩어리 136,847개



,덩어리 수,비율
1일,61625,45.0%
2일,25436,18.6%
3일,12800,9.4%
4일,8045,5.9%
5일,5709,4.2%
6일 이상,23232,17.0%


### 2-3. 사라진 행을 직접 찍어보기

In [39]:
계열이름 = 개인.groupby(S, observed=True).size().reset_index()[S]   # 계열번호 -> 실제 이름표
요일 = calendar.set_index('date')['dow_name']

하루 = 덩어리[덩어리['길이'] == 1].copy()          # 하루짜리 구멍만
i = 하루['계열번호'].values
s = 하루['시작일번호'].values
하루['앞뒤작은값'] = np.minimum(M[i, s - 1], M[i, s + 1])     # 구멍 앞뒤 값 중 작은 쪽

# 앞뒤 값이 큰 것부터, 업종이 겹치지 않게 3개
후보 = 하루.join(계열이름, on='계열번호').sort_values('앞뒤작은값', ascending=False)
예시들 = 후보.drop_duplicates('industry').head(3)

for 순번, (_, r) in enumerate(예시들.iterrows(), start=1):
    계열i, 날짜i = int(r['계열번호']), int(r['시작일번호'])
    키 = 계열이름.iloc[계열i]
    사라진날 = pd.Timestamp(날짜목록[날짜i])

    print('=' * 90)
    print(f"[예시 {순번}] {키['region']} {키['industry']} {키['sex']} {키['age']} {키['time_gb']}"
          f"  ->  {사라진날.date()}({요일[사라진날]})에 행이 사라짐")

    앞, 뒤 = 사라진날 - pd.Timedelta(days=2), 사라진날 + pd.Timedelta(days=2)
    부분 = 개인[(개인['계열번호'] == 계열i) & 개인['date'].between(앞, 뒤)]

    print('원본 5일치')
    display(부분[['date'] + S + ['cnt', 'amount']].sort_values('date').reset_index(drop=True))

    # 5일을 빠짐없이 펼친다. reindex = 없는 날짜 자리를 빈칸(NaN)으로 만들어 준다
    펼침 = 부분.set_index('date')[['cnt', 'amount']].reindex(pd.date_range(앞, 뒤))
    펼침.insert(0, '요일', 요일.reindex(펼침.index).values)
    펼침.insert(1, '행이 있는가', np.where(펼침['cnt'].isna(), '없음  <-- 사라진 날', '있음'))
    print('날짜 펼침')
    display(펼침)

[예시 1] 강남 보험 남성 30대 06_11  ->  2025-12-25(목)에 행이 사라짐
원본 5일치


,date,region,industry,sex,age,time_gb,cnt,amount
0,2025-12-23,강남,보험,남성,30대,06_11,136,64672767
1,2025-12-24,강남,보험,남성,30대,06_11,22420,952633093
2,2025-12-26,강남,보험,남성,30대,06_11,10146,658940293
3,2025-12-27,강남,보험,남성,30대,06_11,23,249973


날짜 펼침


,요일,행이 있는가,cnt,amount
2025-12-23,화,있음,136.00,"64,672,767.00"
2025-12-24,수,있음,"22,420.00","952,633,093.00"
2025-12-25,목,없음 <-- 사라진 날,NaN,NaN
2025-12-26,금,있음,"10,146.00","658,940,293.00"
2025-12-27,토,있음,23.00,"249,973.00"


[예시 2] 강남 인테리어/건축자재/주방기구 여성 50대 06_11  ->  2025-08-19(화)에 행이 사라짐
원본 5일치


...

,요일,행이 있는가,cnt,amount
2025-08-17,일,없음 <-- 사라진 날,NaN,NaN
2025-08-18,월,있음,"1,733.00","51,847,560.00"
2025-08-19,화,없음 <-- 사라진 날,NaN,NaN
2025-08-20,수,있음,"2,512.00","81,563,529.00"
2025-08-21,목,있음,22.00,"834,846.00"


[예시 3] 강남 일반병원 남성 70대 12_17  ->  2025-12-25(목)에 행이 사라짐
원본 5일치


,date,region,industry,sex,age,time_gb,cnt,amount
0,2025-12-23,강남,일반병원,남성,70대,12_17,759,166282478
1,2025-12-24,강남,일반병원,남성,70대,12_17,685,99523469
2,2025-12-26,강남,일반병원,남성,70대,12_17,851,134302021
3,2025-12-27,강남,일반병원,남성,70대,12_17,195,29876797


날짜 펼침


,요일,행이 있는가,cnt,amount
2025-12-23,화,있음,759.00,"166,282,478.00"
2025-12-24,수,있음,685.00,"99,523,469.00"
2025-12-25,목,없음 <-- 사라진 날,NaN,NaN
2025-12-26,금,있음,851.00,"134,302,021.00"
2025-12-27,토,있음,195.00,"29,876,797.00"


셋 다 전날과 다음날에는 있는데 그날만 행이 없다. 사라진 날은 **성탄절, 일요일**이다. 전수 확인은 3장.

## 3. 그게 마스킹인가 - 휴무일로 설명되는가

### 3-1. 판별 아이디어

평소 5~6건 나오던 칸이 하루 0건인 건 이상하지 않다. 문제는 **100건씩 나오던 칸이 통째로 비는** 경우다.
하루짜리 구멍을 앞뒤 값 크기별로 나눠 그날이 **일요일이나 공휴일**이었는지 본다.

In [40]:
# 하루짜리 구멍만 고른다
하루구멍 = 덩어리[덩어리['길이'] == 1].copy()

# 구멍 앞날과 다음날의 값을 꺼낸다.
# (구멍은 관측구간 '안'에만 있으므로 앞날과 다음날은 반드시 존재한다)
i = 하루구멍['계열번호'].values
s = 하루구멍['시작일번호'].values
하루구멍['전날'] = M[i, s - 1]
하루구멍['다음날'] = M[i, s + 1]

# 앞뒤 중 '작은 쪽'을 쓴다. 보수적으로 보기 위해서다.
# (한쪽만 크면 우연히 튄 날일 수 있으니, 양쪽 다 큰 경우만 '큰 계열'로 친다)
하루구멍['앞뒤_작은값'] = 하루구멍[['전날', '다음날']].min(axis=1)

print(f'하루짜리 구멍 {len(하루구멍):,}개')


하루짜리 구멍 61,625개


### 3-2. 달력을 붙여 휴무일인지 확인

In [41]:
# 달력을 날짜 순서에 맞춰 정렬해 둔다
cal = calendar.set_index('date').reindex(pd.to_datetime(날짜목록))
일요일 = (cal['dow'] == 6).values         # dow: 0=월 ... 6=일
공휴일 = cal['is_holiday'].values
휴무일 = 일요일 | 공휴일                   # 일요일이거나 공휴일

하루구멍['휴무일'] = 휴무일[s]

기준선 = 휴무일.mean()    # 184일 중 휴무일 비율. 구멍이 아무 날에나 생긴다면 나와야 할 값

print(f'184일 중 일요일+공휴일 = {휴무일.sum()}일 ({기준선:.1%})')

184일 중 일요일+공휴일 = 33일 (17.9%)


### 3-3. 값 크기별 휴무일 비율

In [42]:
구간 = [(5, 9, '5~9건'), (10, 19, '10~19건'), (20, 49, '20~49건'),
        (50, 99, '50~99건'), (100, 10**9, '100건 이상')]

행 = []
for lo, hi, 이름 in 구간:
    부분 = 하루구멍[(하루구멍['앞뒤_작은값'] >= lo) & (하루구멍['앞뒤_작은값'] <= hi)]
    행.append({'구간': 이름, '하루구멍 수': len(부분), '휴무일 비율': 부분['휴무일'].mean()})

요약 = pd.DataFrame(행).set_index('구간')
보기 = 요약.copy()
보기['하루구멍 수'] = 보기['하루구멍 수'].map('{:,}'.format)
보기['휴무일 비율'] = 보기['휴무일 비율'].map('{:.1%}'.format)
display(보기)


,하루구멍 수,휴무일 비율
구간,,
5~9건,"42,173",23.2%
10~19건,"15,006",30.7%
20~49건,"3,523",54.5%
50~99건,605,87.3%
100건 이상,318,96.5%


### 3-4. 큰 값이 사라진 날은 실제로 무슨 날이었나

In [43]:
# 계열번호 -> 실제 이름표를 되찾는다 (ngroup 순서와 groupby 정렬 순서가 같다)
계열이름 = 개인.groupby(S, observed=True).size().reset_index()[S]

큰구멍 = 하루구멍[하루구멍['앞뒤_작은값'] >= 100].sort_values('앞뒤_작은값', ascending=False)
print(f'앞뒤 모두 100건 이상인데 하루 사라진 경우: {len(큰구멍)}개')
print(f'  그중 휴무일: {int(큰구멍["휴무일"].sum())}개')
print(f'  그중 평일  : {int((~큰구멍["휴무일"]).sum())}개\n')

행 = []
for _, r in 큰구멍.head(10).iterrows():
    i, s = int(r['계열번호']), int(r['시작일번호'])
    k = 계열이름.iloc[i]
    행.append({
        '계열': f"{k['region']} {k['industry']} {k['sex']} {k['age']} {k['time_gb']}",
        '사라진 날': pd.Timestamp(날짜목록[s]).strftime('%Y-%m-%d'),
        '요일': cal['dow_name'].values[s],
        '공휴일': cal['holiday_name'].values[s] or '',
        '전날': int(r['전날']), '다음날': int(r['다음날']),
    })
display(pd.DataFrame(행))

앞뒤 모두 100건 이상인데 하루 사라진 경우: 318개
  그중 휴무일: 307개
  그중 평일  : 11개



,계열,사라진 날,요일,공휴일,전날,다음날
0,강남 보험 남성 30대 06_11,2025-12-25,목,성탄절,22420,10146
1,강남 보험 여성 70대 06_11,2025-12-25,목,성탄절,9885,2057
2,강남 인테리어/건축자재/주방기구 여성 50대 06_11,2025-08-19,화,,1733,2512
3,강남 보험 여성 20대 06_11,2025-12-25,목,성탄절,3095,1420
4,강남 보험 남성 20대 06_11,2025-12-23,화,,1316,2707
5,강남 보험 남성 20대 06_11,2025-12-25,목,성탄절,2707,868
6,강남 일반병원 남성 70대 12_17,2025-12-25,목,성탄절,685,851
7,강남 일반병원 남성 70대 06_11,2025-11-30,일,,581,737
8,춘천 일반병원 남성 60대 06_11,2025-09-28,일,,496,544
9,춘천 일반병원 남성 60대 06_11,2025-10-19,일,,474,582


### 3-5. 평일에 사라진 큰 구멍만 남기면
휴무일로 설명 안 되는 것만 남긴다. **여기 남는 것이 마스킹 후보다.**

In [44]:
행 = []
for th in [20, 50, 100, 200]:
    부분 = 하루구멍[하루구멍['앞뒤_작은값'] >= th]
    평일 = 부분[~부분['휴무일']]
    행.append({'기준': f'앞뒤 {th}건 이상', '하루구멍': len(부분),
               '휴무일': int(부분['휴무일'].sum()), '평일(의심)': len(평일),
               '평일 비율': f'{len(평일) / max(len(부분), 1):.1%}'})
display(pd.DataFrame(행).set_index('기준'))

의심 = 하루구멍[(~하루구멍['휴무일']) & (하루구멍['앞뒤_작은값'] >= 50)]
print(f'끝까지 남는 의심 사례: {len(의심)}개')

,하루구멍,휴무일,평일(의심),평일 비율
기준,,,,
앞뒤 20건 이상,4446,2755,1691,38.0%
앞뒤 50건 이상,923,835,88,9.5%
앞뒤 100건 이상,318,307,11,3.5%
앞뒤 200건 이상,79,76,3,3.8%


끝까지 남는 의심 사례: 88개


## 4. 데이터2로 대조

데이터2에는 시간대 구분이 없다. 그래서 데이터1에서 시간대 칸이 지워져도 **데이터2는 그 물량을 그대로 갖고 있다.**
같은 날 두 파일의 합계를 비교하면 사라진 칸에 뭐가 있었는지 알 수 있다.

```
부족분 = 데이터2 하루 총합 - 데이터1 남은 칸 합
```

### 4-1. 키별로 두 파일의 하루 합계 만들기

In [45]:
KEY = ['date', 'region', 'industry', 'sex', 'age']   # 시간대와 거주지를 뺀 공통 키

# 데이터1 쪽: 키별 합계와, 시간대가 몇 칸 관측됐는지
집계1 = 개인.groupby(KEY, observed=True).agg(
    데이터1_합=('cnt', 'sum'), 관측시간대=('time_gb', 'nunique'))

# 데이터2 쪽: 키별 합계
집계2 = card2.groupby(KEY, observed=True).agg(데이터2_합=('cnt', 'sum'))

대조 = 집계1.join(집계2, how='inner')
대조['부족분'] = 대조['데이터2_합'] - 대조['데이터1_합']   # 양수 = 데이터1에 없는 물량

print(f'두 파일에 모두 있는 키 {len(대조):,}개')
print(f'데이터1 총합 {대조["데이터1_합"].sum():,}')
print(f'데이터2 총합 {대조["데이터2_합"].sum():,}')

두 파일에 모두 있는 키 371,734개
데이터1 총합 571,960,474
데이터2 총합 571,833,984


### 4-2. 기준선: 이 정도 차이는 원래 있다
빠진 칸이 **하나도 없는 키**에서도 두 파일 합계가 어긋나는지 본다.

In [46]:
빈칸있음 = 대조[대조['관측시간대'] < 4]['부족분']     # 시간대 칸이 1~3개만 있는 키
빈칸없음 = 대조[대조['관측시간대'] == 4]['부족분']    # 4칸이 다 있는 키 (빠진 게 없다)

비교 = pd.DataFrame({
    '빈 칸이 있는 키': 빈칸있음.describe(percentiles=[.01, .5, .99]),
    '빈 칸이 없는 키 (기준선)': 빈칸없음.describe(percentiles=[.01, .5, .99]),
}).round(2)
display(비교)

print(f'기준선 키: 0이 아닌 비율 {(빈칸없음 != 0).mean():.1%}, '
      f'폭 {빈칸없음.min():.0f}~{빈칸없음.max():.0f}, '
      f'5 이상 {int((빈칸없음 >= 5).sum()):,}개 / {len(빈칸없음):,}개')

,빈 칸이 있는 키,빈 칸이 없는 키 (기준선)
count,"284,677.00","87,057.00"
mean,-0.27,-0.57
std,0.83,1.24
min,-6.00,-6.00
1%,-3.00,-3.00
50%,0.00,-1.00
99%,1.00,2.00
max,4.00,6.00


기준선 키: 0이 아닌 비율 69.5%, 폭 -6~6, 5 이상 3개 / 87,057개


빠진 칸이 없는 키에서도 똑같이 어긋난다. **±1~3은 빈 칸과 무관한 배경 차이**이므로 지워진 흔적으로 읽을 수 없다.

### 4-3. 의심 사례 88개 전수 대조

| 부족분 | 뜻 |
|---|---|
| **5 이상** | 사라진 칸에 결제가 있었다 = 지워진 것 |
| **5 미만** | 사라진 칸은 비어 있었다 = 그날 0건 (남는 ±1~3은 4-2) |

In [47]:
# 의심 사례에 '어느 키의 며칠인지'를 붙인다
의심키 = 의심.join(계열이름, on='계열번호').copy()
의심키['date'] = pd.to_datetime(날짜목록[의심키['시작일번호'].values])

# 두 파일의 키별 하루 합계를 각각 가져온다
확인 = (의심키.merge(집계1.reset_index(), on=KEY, how='left')
              .merge(집계2.reset_index(), on=KEY, how='left'))

# 키가 아예 없다 = 그 파일 기준으로 그날 0건이라는 뜻이다. 0으로 채운다.
#   데이터1에 없는 경우 = 그날 그 키의 시간대 칸이 하나도 없었다
#   데이터2에 없는 경우 = 데이터2 기준으로도 그날 결제가 없었다
데이터1없음 = 확인['데이터1_합'].isna()
데이터2없음 = 확인['데이터2_합'].isna()
확인[['데이터1_합', '데이터2_합']] = 확인[['데이터1_합', '데이터2_합']].fillna(0)
확인['부족분'] = 확인['데이터2_합'] - 확인['데이터1_합']
print(f'의심 사례 {len(확인)}개 대조 (그날 데이터1 칸이 전부 없던 사례 {데이터1없음.sum()}개 포함)')
print(f"  부족분 최소 {확인['부족분'].min():.0f} / 중앙값 {확인['부족분'].median():.0f} / 최대 {확인['부족분'].max():.0f}")
display(확인['부족분'].astype(int).value_counts().sort_index().rename('의심 사례 수').to_frame().T)

지워진흔적 = int((확인['부족분'] >= 5).sum())
print(f'부족분 5 이상(= 지워진 흔적): {지워진흔적}건 / {len(확인)}건')

if 데이터2없음.any():
    print(f'\n데이터2에도 키가 없던 사례 {데이터2없음.sum()}개 (두 파일 모두 그날 0건)')
    display(확인.loc[데이터2없음, ['date', 'region', 'industry', 'sex', 'age', 'time_gb',
                                  '앞뒤_작은값', '데이터1_합', '데이터2_합', '부족분']])

의심 사례 88개 대조 (그날 데이터1 칸이 전부 없던 사례 5개 포함)
  부족분 최소 -3 / 중앙값 0 / 최대 1


부족분,-3,-2,-1,0,1
의심 사례 수,1,5,22,52,8


부족분 5 이상(= 지워진 흔적): 0건 / 88건

데이터2에도 키가 없던 사례 5개 (두 파일 모두 그날 0건)


,date,region,industry,sex,age,time_gb,앞뒤_작은값,데이터1_합,데이터2_합,부족분
4,2025-08-07,강남,게임방/오락실,여성,70대,12_17,130.00,0.00,0.00,0.00
5,2025-07-12,강남,게임방/오락실,여성,80대,18_23,54.00,0.00,0.00,0.00
12,2025-12-23,강남,보험,여성,80대,06_11,127.00,0.00,0.00,0.00
47,2025-08-25,춘천,문화용품,남성,40대,12_17,52.00,0.00,0.00,0.00
48,2025-07-28,춘천,문화용품,여성,30대,12_17,83.00,0.00,0.00,0.00


### 4-4. 값이 큰 10개

In [48]:
상세 = 확인.sort_values('앞뒤_작은값', ascending=False).head(10)
요일이름 = calendar.set_index('date')['dow_name']

표 = pd.DataFrame({
    '계열': (상세['region'] + ' ' + 상세['industry'] + ' ' + 상세['sex'] + ' ' + 상세['age']).values,
    '사라진 칸': 상세['time_gb'].values,
    '날짜': (상세['date'].dt.strftime('%Y-%m-%d') + '(' + 상세['date'].map(요일이름) + ')').values,
    '전날/다음날 중 작은 값': 상세['앞뒤_작은값'].astype(int).values,
    '데이터1 남은 칸 합': 상세['데이터1_합'].astype(int).values,
    '데이터2 하루 총합': 상세['데이터2_합'].astype(int).values,
    '차이': 상세['부족분'].astype(int).values,
})
display(표)

,계열,사라진 칸,날짜,전날/다음날 중 작은 값,데이터1 남은 칸 합,데이터2 하루 총합,차이
0,강남 인테리어/건축자재/주방기구 여성 50대,06_11,2025-08-19(화),1733,60,59,-1
1,강남 보험 남성 20대,06_11,2025-12-23(화),1316,49,47,-2
2,춘천 완구/아동용자전거 남성 40대,18_23,2025-08-13(수),254,69,69,0
3,춘천 완구/아동용자전거 남성 20대,00_05,2025-09-11(목),176,367,367,0
4,춘천 완구/아동용자전거 남성 20대,12_17,2025-09-04(목),144,57,57,0
5,강남 보험 여성 80대,06_11,2025-09-23(화),138,5,5,0
6,춘천 완구/아동용자전거 여성 20대,00_05,2025-09-16(화),132,361,362,1
7,강남 게임방/오락실 여성 70대,12_17,2025-08-07(목),130,0,0,0
8,강남 보험 여성 80대,06_11,2025-12-23(화),127,0,0,0
9,춘천 완구/아동용자전거 여성 20대,00_05,2025-11-12(수),116,480,481,1


## 5. 결론

| | |
|---|---|
| 있던 행이 다음 날 사라지는 사례 | 136,847건 |
| 앞뒤 100건 이상인 큰 구멍 | 318개, 그중 **307개(96.5%)가 일요일/공휴일** (3-3) |
| 휴무일로 설명 안 되는 의심 사례 | **88개** (3-5) |
| 그 88개의 부족분 | 전부 5 미만. **마스킹 흔적 0건** (4-3) |

**사라지는 행은 흔하지만 전부 그날 결제가 0건이었던 칸이다. 따라서 마스킹의 증거는 없다.**